# Hyperparameter tuning sa RandomizedSearchCV metodom

Model: **SVC**

Dataset: **Iris**

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import f1_score

In [12]:
# Ucitavanje dataseta i podjela na trening i test skupove
df = pd.read_csv("iris.csv") 
X = df.drop('species', axis=1)
y = df['species']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

#### Izbor parametara sa RandomizedSearchCV

In [13]:
OPCIJE = {
    # "random_state": np.arange(0, 50).tolist(), - izbaceno radi reproducibilnosti
    "random_state": [0],
    "C": np.arange(0.1, 10.1, 0.1).tolist(),
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    "break_ties": [True, False],
    "cache_size": [200, 300, 400],
    "class_weight": [None, 'balanced'],
    "coef0": [0.0, 0.1, 0.5, 1.0],
    # izbacen 'ovo' jer ako je break ties True desava se warning
    "decision_function_shape": ['ovr'],
    "degree": [3, 4, 5],
    # povecane cifre u odnosu na vjezbe radi bolje konvergencije
    "max_iter": [-1, 1000, 2000],
    "probability": [True, False],
    "shrinking": [True, False],
    "tol": [1e-3, 1e-4, 1e-5],
    "verbose": [0, 1]
    #sample
}

random_search = RandomizedSearchCV(
    estimator=SVC(),
    param_distributions=OPCIJE,
    n_iter=100,
    scoring='f1_weighted',
    random_state=0
)
random_search.fit(X_train, y_train)
random_search.best_params_, random_search.best_score_

[LibSVM]*
optimization finished, #iter = 8
obj = -4.373101, rho = -0.138659
nSV = 10, nBSV = 8
*
optimization finished, #iter = 11
obj = -2.073612, rho = 0.050721
nSV = 6, nBSV = 3
*
optimization finished, #iter = 24
obj = -20.776384, rho = 0.029128
nSV = 37, nBSV = 34
Total nSV = 48
[LibSVM]*
optimization finished, #iter = 16
obj = -4.329298, rho = -0.132359
nSV = 11, nBSV = 8
*
optimization finished, #iter = 21
obj = -1.857669, rho = 0.013892
nSV = 7, nBSV = 3
*
optimization finished, #iter = 20
obj = -19.284437, rho = 0.105272
nSV = 33, nBSV = 31
Total nSV = 45
[LibSVM]*
optimization finished, #iter = 16
obj = -3.980798, rho = -0.025724
nSV = 11, nBSV = 8
*
optimization finished, #iter = 22
obj = -2.050223, rho = 0.089569
nSV = 8, nBSV = 3
*
optimization finished, #iter = 26
obj = -22.132834, rho = 0.070618
nSV = 38, nBSV = 35
Total nSV = 50
[LibSVM]*
optimization finished, #iter = 16
obj = -3.847171, rho = -0.153959
nSV = 10, nBSV = 6
*
optimization finished, #iter = 36
obj = -1.98

({'verbose': 1,
  'tol': 0.001,
  'shrinking': True,
  'random_state': 0,
  'probability': False,
  'max_iter': -1,
  'kernel': 'rbf',
  'gamma': 'auto',
  'degree': 5,
  'decision_function_shape': 'ovr',
  'coef0': 1.0,
  'class_weight': 'balanced',
  'cache_size': 300,
  'break_ties': False,
  'C': 1.3000000000000003},
 np.float64(0.9663763166085146))

#### Predikcije pomocu odabranih parametara i evaluiranje f1 score-a

In [14]:
final_model = random_search.best_estimator_
predictions = final_model.predict(X_test)
f1_score(y_test, predictions, average='weighted')

1.0

#### Poređenje sa SVC modelom sa "nasumično" odabranim parametrima

In [15]:
random_SVC = SVC(
    C=0.1,
    kernel="sigmoid",
    gamma="auto",
    random_state=0,
)
random_SVC.fit(X_train, y_train)
y_pred = random_SVC.predict(X_test)
f1_score(y_test, y_pred, average='weighted')

0.06666666666666667